# Dimension 2: Runtime and performance
## Evaluation setup


## 1) Step

Create scaled data sets for performance test.

In [0]:
# CREATE SCALED DATA SETS FOR PERFORMANCE TEST 
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# Source: TPC-DS Samples in Databricks
source_db = "samples.tpcds_sf1000"

# Target
catalog_schema = "`eva-dev-psc-we-main-enriched_catalog`.sdp_poc"

# Tables
tables = [
    "store_sales",
    "date_dim",
    "item",
    "store",
    "customer"
]

# Scaling factors
factors = [2]

def make_scaled_df(df: DataFrame, factor: int) -> DataFrame:
    """
    creates table x factor
    """
    dfs = []
    for i in range(factor):
        dfs.append(
            df.withColumn("scale_factor_copy", F.lit(i + 1))
        )
    return reduce(DataFrame.unionByName, dfs)

for table in tables:
    source_table = f"{source_db}.{table}"
    df_base = spark.table(source_table)

    for f in factors:
        df_scaled = make_scaled_df(df_base, f)

        # target table name
        target_table_name = f"{table}_x{f}"
        full_name = f"{catalog_schema}.{target_table_name}"

        (
            df_scaled.write
            .mode("overwrite")          
            .format("delta")
            .saveAsTable(full_name)
        )

        print(f"Geschrieben: {full_name}")

## 2) Step

Delete all existing tables that are overwritten by the pipelines to ensure comparable results.
Choose the notebook based on where you are testing. You may need to adjust the catalog and schema names.

In [0]:
# EXECUTE BEFORE EACH PIPLINE RUN IF TESTING IN PSC DEV
schema_name = "`eva-dev-psc-we-main-enriched_catalog`.sdp_poc"

# list all tables in schema
tables_df = spark.sql(f"SHOW TABLES IN {schema_name}")

# delete tables starting with i_ or d_ (= deletes all existing tables from the last pipeline runs)
for row in tables_df.collect():
    table_name = row["tableName"]
    if table_name.startswith("i_") or table_name.startswith("d_"):
        table_full_name = f"{schema_name}.`{table_name}`"
        print(f"Dropping table: {table_full_name}")
        spark.sql(f"DROP TABLE IF EXISTS {table_full_name}")


In [0]:
# EXECUTE BEFORE EACH DECLARATIVE PIPLINE RUN IF TESTING IN PRIVATE DATABRICKS ACCOUNT
schema_name = "workspace.declarative"  
tables_df = spark.sql(f"SHOW TABLES IN {schema_name}")

# delete all tables in schema
for row in tables_df.collect():
    table_full_name = f"{schema_name}.{row['tableName']}"
    spark.sql(f"DROP TABLE IF EXISTS {table_full_name}")

In [0]:
# EXECUTE BEFORE EACH IMPERATIVE PIPLINE RUN  IF TESTING IN PRIVATE DATABRICKS ACCOUNT
schema_name = "workspace.imperative"  
tables_df = spark.sql(f"SHOW TABLES IN {schema_name}")

# delete all tables in schema
for row in tables_df.collect():
    table_full_name = f"{schema_name}.{row['tableName']}"
    spark.sql(f"DROP TABLE IF EXISTS {table_full_name}")